In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.etabs_api import start_api, create_dp

SapModel = start_api(verbose=True)

group_name = "OPT_20"
mat_names = ["C30/37 Zone 1", "C30/37 Zone 2", "C30/37 Zone 3"]
E_i = 32836.6
n_elements = len(mat_names)

In [ ]:
from src.convergence import run_convergence_sweep, compute_convergence_errors

# --- Configuration ---
modes_range = range(1, 16)   # Test 1 through 15 modes
baseline_n = 30              # Massive mode count for the "Truth" matrix
target_case = "Modal"        # Must match your ETABS modal case name exactly

# --- Run the sweep (handles unlock → set modes → analyse → extract → time) ---
flexibility_matrices, compute_times = run_convergence_sweep(
    SapModel, group_name, E_i, mat_names,
    modes_range=modes_range,
    baseline_n=baseline_n,
    modal_case=target_case,
)

# --- Compute relative Frobenius-norm error vs. the massive baseline "truth" ---
convergence_errors = compute_convergence_errors(
    flexibility_matrices, modes_range, reference_n=baseline_n
)

print("Data collection complete!")
print(f"Errors (%): {[f'{e:.2f}' for e in convergence_errors]}")
print(f"Times  (s): {[f'{t:.1f}' for t in compute_times]}")

In [ ]:
from src.convergence import find_elbow, plot_convergence_vs_cost
import matplotlib.pyplot as plt

# --- Find the optimal trade-off point (Kneedle / max-distance method) ---
elbow_n, diagnostics = find_elbow(modes_range, convergence_errors, compute_times)
print(f"Optimal elbow point: n = {elbow_n} modes")
print(f"  → Error at elbow : {convergence_errors[elbow_n - 1]:.2f} %")
print(f"  → Time  at elbow : {compute_times[elbow_n - 1]:.1f} s")

# --- Generate the publication-quality Pareto trade-off figure ---
fig, (ax1, ax2) = plot_convergence_vs_cost(
    modes_range,
    convergence_errors,
    compute_times,
    elbow_n=elbow_n,
    save_path='../data/processed/convergence_vs_cost.png',
)

plt.show()